In [ ]:
import os
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.cm as cm
import matplotlib.patches as mpatches
import matplotlib.pyplot as plt
from matplotlib.patches import Patch

# Path-neutral configuration. Set these before running the notebook.
DATA_ROOT = Path(os.environ["AMAZON_DATA_DIR"])
OUTPUT_ROOT = Path(os.environ["AMAZON_OUTPUT_DIR"])

MODELS_ROOT = DATA_ROOT / "intermediate" / "06_sdm" / "models"
SPECIES_LIST_FILE = (
    DATA_ROOT / "intermediate" / "04_rasterize_species"
    / "full_species_list_amazon_updated.csv"
)
OUTPUT_DIR = OUTPUT_ROOT / "08_figures" / "appendix" / "tss_scores"
DATA_DIR = OUTPUT_DIR / "data"
PLOT_DIR = OUTPUT_DIR / "plots" / "model_performance"
DATA_DIR.mkdir(parents=True, exist_ok=True)
PLOT_DIR.mkdir(parents=True, exist_ok=True)

ALGORITHMS = ["GLM", "GAM", "GBM", "RFd"]
RANGE_CLASSES = ["supersmall", "small", "medium", "large", "superlarge"]

In [ ]:
def require_files(*paths):
    missing = [str(path) for path in paths if not Path(path).is_file()]
    if missing:
        raise FileNotFoundError("Missing required file(s):\n" + "\n".join(missing))


def normalise_species_name(names):
    """Use one matching key for dots, hyphens, and underscores in species names."""
    return (
        names.astype("string")
        .str.replace(r"[-.]", "_", regex=True)
        .str.replace(r"_+", "_", regex=True)
    )


def attach_range_size_class(performance_df, species_data, species_column="Species"):
    performance = performance_df.copy()
    performance["_species_key"] = normalise_species_name(performance[species_column])

    # SDMs include only resident birds; mammals, amphibians, and reptiles are
    # included irrespective of seasonality. This avoids duplicate bird names
    # from breeding/non-breeding entries in the full species list.
    is_modelled_taxon = (
        species_data["type"].ne("birds")
        | species_data["seasonality"].eq("resident")
    )
    species_lookup = species_data.loc[
        is_modelled_taxon,
        ["Species", "class"],
    ].copy()
    species_lookup["_species_key"] = normalise_species_name(species_lookup["Species"])
    duplicate_keys = species_lookup["_species_key"].duplicated(keep=False)
    if duplicate_keys.any():
        duplicates = species_lookup.loc[duplicate_keys, "Species"].tolist()
        raise ValueError(
            "Species-name normalisation produces duplicate lookup keys: "
            + ", ".join(duplicates)
        )

    merged = performance.merge(
        species_lookup[["_species_key", "class"]],
        on="_species_key",
        how="left",
    ).drop(columns="_species_key")

    missing_classes = merged["class"].isna().sum()
    if missing_classes:
        print(f"Warning: {missing_classes} performance rows have no range-size class.")
    return merged


def plot_calibration_validation_by_algorithm(performance_df, output_path, title=None):
    grouped_calibration = performance_df.groupby("algo")["calibration"]
    grouped_validation = performance_df.groupby("algo")["validation"]
    missing_algorithms = [
        algorithm for algorithm in ALGORITHMS
        if algorithm not in grouped_calibration.groups or algorithm not in grouped_validation.groups
    ]
    if missing_algorithms:
        raise ValueError("Missing algorithm(s): " + ", ".join(missing_algorithms))

    fig, ax = plt.subplots(figsize=(10, 5))
    box_width = 0.25
    separation = 0.15
    positions = np.arange(len(ALGORITHMS))

    for index, algorithm in enumerate(ALGORITHMS):
        ax.boxplot(
            grouped_calibration.get_group(algorithm).dropna(),
            positions=[positions[index] - separation],
            widths=box_width,
            patch_artist=True,
            showfliers=False,
            boxprops=dict(facecolor="lightgreen"),
        )
        ax.boxplot(
            grouped_validation.get_group(algorithm).dropna(),
            positions=[positions[index] + separation],
            widths=box_width,
            patch_artist=True,
            showfliers=False,
            boxprops=dict(facecolor="forestgreen"),
        )

    ax.set_xticks(positions)
    ax.set_xticklabels(ALGORITHMS, fontsize=16, fontweight="bold")
    ax.set_ylabel("TSS", fontsize=16, fontweight="bold")
    ax.tick_params(axis="y", labelsize=14)
    ax.set_ylim(0, 1)
    if title:
        ax.set_title(title, fontsize=16, fontweight="bold")
    ax.legend(
        handles=[
            Patch(facecolor="lightgreen", label="Calibration"),
            Patch(facecolor="forestgreen", label="Validation"),
        ],
        loc="lower right",
    )
    fig.tight_layout()
    fig.savefig(output_path, dpi=300, bbox_inches="tight")
    plt.show()


def lighten(color, factor=0.55):
    red, green, blue, alpha = color
    return (1 - factor) + factor * red, (1 - factor) + factor * green, (1 - factor) + factor * blue, alpha


def darken(color, factor=0.55):
    red, green, blue, alpha = color
    return factor * red, factor * green, factor * blue, alpha


def plot_calibration_validation_by_algorithm_and_class(performance_df, output_path, title=None):
    cmap = cm.get_cmap("Dark2")
    base_colors = [cmap(value) for value in np.linspace(0.15, 0.95, len(RANGE_CLASSES))]
    calibration_colors = [lighten(color) for color in base_colors]
    validation_colors = [darken(color) for color in base_colors]

    fig, ax = plt.subplots(figsize=(22, 6))
    boxes_per_algorithm = len(RANGE_CLASSES) * 2
    algorithm_positions = np.arange(len(ALGORITHMS)) * (boxes_per_algorithm + 2)
    box_data, positions, box_colors = [], [], []

    for algorithm_index, algorithm in enumerate(ALGORITHMS):
        algorithm_data = performance_df[performance_df["algo"] == algorithm]
        start = algorithm_positions[algorithm_index]
        for class_index, range_class in enumerate(RANGE_CLASSES):
            class_data = algorithm_data[algorithm_data["class"] == range_class]
            box_data.extend([
                class_data["calibration"].dropna(),
                class_data["validation"].dropna(),
            ])
            positions.extend([start + class_index * 2, start + class_index * 2 + 1])
            box_colors.extend([calibration_colors[class_index], validation_colors[class_index]])

    boxplot = ax.boxplot(
        box_data,
        positions=positions,
        widths=0.6,
        patch_artist=True,
        showfliers=False,
    )
    for box, color in zip(boxplot["boxes"], box_colors):
        box.set_facecolor(color)

    ax.set_xticks(algorithm_positions + boxes_per_algorithm / 2 - 0.5)
    ax.set_xticklabels(ALGORITHMS, fontsize=16, fontweight="bold")
    ax.set_ylabel("TSS", fontsize=16, fontweight="bold")
    ax.tick_params(axis="y", labelsize=14)
    ax.set_ylim(0, 1)
    if title:
        ax.set_title(title, fontsize=16, fontweight="bold")

    legend_handles, legend_labels = [], []
    for range_class, calibration_color, validation_color in zip(
        RANGE_CLASSES, calibration_colors, validation_colors
    ):
        legend_handles.extend([
            mpatches.Patch(facecolor=(0, 0, 0, 0), edgecolor=(0, 0, 0, 0)),
            mpatches.Patch(facecolor=calibration_color),
            mpatches.Patch(facecolor=validation_color),
        ])
        legend_labels.extend([
            f"$\\mathbf{{{range_class}}}$", "Calibration", "Validation"
        ])

    fig.subplots_adjust(right=0.8)
    ax.legend(
        legend_handles,
        legend_labels,
        loc="center left",
        bbox_to_anchor=(1.01, 0.5),
        fontsize=12,
        frameon=True,
        borderpad=1.2,
        labelspacing=0.6,
    )
    fig.savefig(output_path, dpi=300, bbox_inches="tight")
    plt.show()

In [ ]:
# RandomCV: collect final-model calibration and validation TSS from all species.
require_files(SPECIES_LIST_FILE)
species_data = pd.read_csv(SPECIES_LIST_FILE)

randomcv_results = []
for species_folder in sorted(MODELS_ROOT.iterdir()):
    if not species_folder.is_dir():
        continue
    evaluation_file = (
        species_folder / "final_model_evaluation" / "final_model_tuned.csv"
    )
    if not evaluation_file.is_file():
        print(f"Missing RandomCV evaluation: {species_folder.name}")
        continue
    randomcv_results.append(pd.read_csv(evaluation_file))

if not randomcv_results:
    raise RuntimeError(f"No RandomCV evaluation files found under {MODELS_ROOT}")

randomcv_performance = pd.concat(randomcv_results, ignore_index=True)
randomcv_performance["Species"] = (
    randomcv_performance["full.name"]
    .astype(str)
    .str.split("_", n=1).str[0]
    .str.replace(".", "_", regex=False)
)
randomcv_performance = attach_range_size_class(randomcv_performance, species_data)

randomcv_data_file = DATA_DIR / "final_model_evaluation_all_species_randomCV.csv"
randomcv_performance.to_csv(randomcv_data_file, index=False)
print(f"RandomCV performance rows: {len(randomcv_performance)}")
print(f"Saved: {randomcv_data_file}")

In [ ]:
# RandomCV: calibration and validation TSS by algorithm.
plot_calibration_validation_by_algorithm(
    randomcv_performance,
    PLOT_DIR / "validation_calibration_TSS_randomCV.png",
)

In [ ]:
# RandomCV: calibration and validation TSS by algorithm and range-size class.
plot_calibration_validation_by_algorithm_and_class(
    randomcv_performance,
    PLOT_DIR / "calibration_validation_TSS_randomCV_per_algorithm.png",
)

In [ ]:
# SpatialCV: for each species and algorithm, retain all runs of the best
# hyperparameter combination (defined by mean validation TSS across spatial folds).
non_hyperparameter_columns = {
    "full.name", "PA", "run", "algo", "metric.eval", "cutoff",
    "sensitivity", "specificity", "calibration", "validation", "evaluation",
}
spatialcv_results = []

for species_folder in sorted(MODELS_ROOT.iterdir()):
    if not species_folder.is_dir():
        continue
    species_name = species_folder.name.replace(".", "_")
    tuning_directory = species_folder / "models_evaluation_blockCV_tuning"
    if not tuning_directory.is_dir():
        print(f"Missing SpatialCV tuning directory: {species_folder.name}")
        continue

    for algorithm in ALGORITHMS:
        tuning_file = tuning_directory / algorithm / f"{algorithm}_summary_tuning.csv"
        if not tuning_file.is_file():
            print(f"Missing SpatialCV tuning file: {tuning_file}")
            continue

        tuning_data = pd.read_csv(tuning_file)
        hyperparameter_columns = [
            column for column in tuning_data.columns
            if column not in non_hyperparameter_columns
        ]
        if not hyperparameter_columns:
            raise ValueError(f"No hyperparameter columns found in {tuning_file}")

        mean_validation = (
            tuning_data.groupby(hyperparameter_columns, dropna=False)["validation"]
            .mean()
            .reset_index()
        )
        best_hyperparameters = mean_validation.loc[mean_validation["validation"].idxmax()]
        is_best = (
            tuning_data[hyperparameter_columns]
            .eq(best_hyperparameters[hyperparameter_columns])
            .all(axis=1)
        )
        best_runs = tuning_data.loc[
            is_best,
            ["full.name", "run", "algo", "sensitivity", "specificity", "calibration", "validation"],
        ].copy()
        best_runs["Species"] = species_name
        spatialcv_results.append(best_runs)

if not spatialcv_results:
    raise RuntimeError(f"No SpatialCV tuning files found under {MODELS_ROOT}")

spatialcv_performance = pd.concat(spatialcv_results, ignore_index=True)
spatialcv_performance = attach_range_size_class(spatialcv_performance, species_data)

spatialcv_data_file = DATA_DIR / "best_hyperparameters_all_species_spatialCV.csv"
spatialcv_performance.to_csv(spatialcv_data_file, index=False)
print(f"SpatialCV performance rows: {len(spatialcv_performance)}")
print(f"Saved: {spatialcv_data_file}")

In [ ]:
# SpatialCV: calibration and validation TSS by algorithm.
plot_calibration_validation_by_algorithm(
    spatialcv_performance,
    PLOT_DIR / "calibration_validation_TSS_spatialCV.png",
)

In [ ]:
# SpatialCV: calibration and validation TSS by algorithm and range-size class.
plot_calibration_validation_by_algorithm_and_class(
    spatialcv_performance,
    PLOT_DIR / "calibration_validation_TSS_spatialCV_per_algorithm.png",
)